# Cash Flow Engine and Field Economics


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch20_production_optimisation\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch20_production_optimisation\figures
NeqSim Java bridge available: True


## Production Decline Models


In [2]:
years = np.arange(1, 21)
peak = 55.0
exponential = peak * np.exp(-0.13 * (years - 1))
hyperbolic = peak / np.power(1 + 0.45 * (years - 1), 1 / 1.2)
plateau_decline = np.where(years <= 5, peak, peak * np.exp(-0.18 * (years - 5)))
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(years, exponential, label="Exponential")
ax.plot(years, hyperbolic, label="Hyperbolic")
ax.plot(years, plateau_decline, label="Plateau then decline")
ax.set_xlabel("Production year")
ax.set_ylabel("Gas production (MSm3/d equivalent)")
ax.set_title("Production Profiles for Economic Evaluation")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch20_economics_production_profiles.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch20_economics_production_profiles.png).** *Observation.* The plateau-then-decline profile keeps early revenue high while preserving a long decline tail. *Mechanism.* Facility capacity caps early production before reservoir deliverability controls late-life rates. *Implication.* NPV is strongly affected by plateau duration because discounting favors early cash flow. *Recommendation.* Show at least one production-profile sensitivity with every concept NPV.


## Discounted Cash Flow


In [3]:
capex = np.array([900, 850, 350] + [0] * 18, dtype=float)
production = np.array([0, 0, 20] + list(plateau_decline[:-2]))
revenue = production * 365 * 1.55
opex = np.where(production > 0, 165, 0)
pretax_cf = revenue - opex - capex
discount_rate = 0.08
discount = np.power(1 + discount_rate, -np.arange(len(pretax_cf)))
discounted = pretax_cf * discount
npv = discounted.sum()
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(np.arange(len(pretax_cf)), pretax_cf, color=np.where(pretax_cf >= 0, "#2a9d8f", "#e76f51"), alpha=0.75, label="Annual cash flow")
ax.plot(np.cumsum(discounted), color="#264653", linewidth=2, label="Cumulative discounted CF")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Project year")
ax.set_ylabel("MNOK")
ax.set_title(f"Discounted Cash Flow, NPV = {npv:.0f} MNOK")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch20_economics_discounted_cash_flow.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"NPV at {discount_rate:.0%}: {npv:.0f} MNOK")


NPV at 8%: 172533 MNOK


**Discussion (Figure ch20_economics_discounted_cash_flow.png).** *Observation.* The project is cash-negative during development and recovers after production reaches plateau. *Mechanism.* Front-loaded CAPEX is discounted less than late-life production, so schedule and first-gas timing dominate value. *Implication.* A delayed startup can reduce NPV even when ultimate recovery is unchanged. *Recommendation.* Report NPV together with first-production date and plateau duration.


## NPV Sensitivity


In [4]:
gas_prices = np.linspace(1.0, 2.2, 9)
npvs = []
for price in gas_prices:
    revenue_i = production * 365 * price
    cf_i = revenue_i - opex - capex
    npvs.append((cf_i * discount).sum())
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(gas_prices, npvs, marker="o", color="#1d3557")
ax.axhline(0, color="#e76f51", linestyle="--")
ax.set_xlabel("Gas price (NOK/Sm3 equivalent)")
ax.set_ylabel("NPV (MNOK)")
ax.set_title("Gas-Price Sensitivity")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURES_DIR / "ch20_economics_npv_sensitivity.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch20_economics_npv_sensitivity.png).** *Observation.* NPV crosses zero between the low and base gas-price cases. *Mechanism.* Revenue scales almost linearly with price while most CAPEX is fixed. *Implication.* Market assumptions can dominate concept ranking once technical feasibility is established. *Recommendation.* Use P10/P50/P90 price assumptions and disclose breakeven price in the decision summary.
